# Episode 6 — GPT-2 Transformer in Pure JAX

**Student workbook** · code along with the video.

Build a **decoder-only GPT-2** transformer from scratch: parameter **PyTrees** (`NamedTuple`), forward pass, cross-entropy loss, and **plain SGD** updates — no Flax, no Optax.

This episode mirrors [`transformer.py`](../transformer.py) in the repo root. Tensor axes use **Noam suffix notation**: `b` batch, `s` sequence, `h` hidden, `f` FFN dim (`mlp_mult * H`), `v` vocab, `p` heads, `d` head dim, `i`/`o` matmul axes.

| | |
|---|---|
| **Chapter** | 2.1 · Part II — GPT-2 Transformer |
| **Prereq** | Episodes 1–5 (especially pytrees and `grad`) |
| **Next** | Part II — training at scale |

**References:** [GPT-2 paper](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) · [JAX pytrees](https://docs.jax.dev/en/latest/pytrees.html)


## Imports and environment

Standard imports for the episode. `TF_CPP_MIN_LOG_LEVEL=2` silences noisy XLA GPU autotuning logs — set it **before** importing JAX.


In [ ]:
# your code here


## Config

`GPTConfig` holds model hyperparameters. `gpt2_small()` returns the **124M-parameter** GPT-2 Small defaults (12 layers, 768 hidden, 12 heads, ctx 1024).

The demo at the end uses a **tiny** config so the notebook trains in reasonable time on a single GPU.


### Config types

Define the config NamedTuple, GPT-2 Small preset, and a tiny demo config.

### `GPTConfig, gpt2_small, demo_config`

| | |
|---|---|
| **`GPTConfig`** | fields: `vocab_dim`, `max_ctx`, `n_layers`, `n_heads`, `hidden`, `mlp_mult` |
| **`gpt2_small(vocab_dim=50257)`** | returns 124M-param GPT-2 Small defaults |
| **`demo_config(vocab_dim)`** | returns tiny config (2 layers, 64 hidden) for notebook demos |
| **returns** | `GPTConfig` NamedTuple; two helper functions returning `GPTConfig` |

**Implement in the code cell below.**


In [ ]:
# your code here


## Parameter PyTrees

All learnable weights live in **NamedTuples** so `jax.grad`, `jax.tree.map`, and `jax.jit` work without custom registration (Episode 5).

| Type | Fields | Shapes |
|------|--------|--------|
| `LayerNormParams` | `gamma_h`, `beta_h` | `(H,)`, `(H,)` |
| `LinearParams` | `w_io`, `b_o` | `(I, O)`, `(O,)` |
| `MultiHeadAttentionParams` | `proj_q/k/v/o` | each `LinearParams (H, H)` |
| `MLPParams` | `fc_hf`, `fc_fh` | `(H, F)`, `(F, H)` |
| `BlockParams` | `ln1_h`, `attn_hh`, `ln2_h`, `mlp_hh` | one pre-LN block |
| `TransformerParams` | `embed_vh`, `pos_embed_sh`, `blocks_l`, `ln_final_h` | token + position embeds, stack of blocks |


### Parameter types

Declare every parameter container as a `NamedTuple`. No methods — just typed fields.

### `LayerNormParams, LinearParams, MultiHeadAttentionParams, MLPParams, BlockParams, TransformerParams`

| | |
|---|---|
| *(fields)* | see shape table above |
| **returns** | the six NamedTuple classes |

**Implement in the code cell below.**


In [ ]:
# your code here


## Initialisation and forward pass

GPT-2 style init: linear weights \(\mathcal{N}(0, 0.02^2)\), token embeddings `* 0.02`, position embeddings `* 0.01`, layer-norm `gamma=1` / `beta=0`, linear biases `0`.

For each submodule we define **init** then **forward** in the same place: linear → layer norm → attention → MLP → block → full model.

Pre-LN GPT-2 block: layer norm before attention and MLP; residual connections wrap each sublayer.

Split PRNG keys per submodule (Episode 1).


### Linear weight init

Sample Gaussian weights and zero biases.

### `_linear(key: Array, n_in: int, n_out: int, std: float = 0.02) -> LinearParams`

| | |
|---|---|
| **`key`** | PRNG key |
| **`n_in`** | input features `I` |
| **`n_out`** | output features `O` |
| **`std`** | Gaussian scale (default `0.02`) |
| **returns** | `LinearParams(w_io=(I, O), b_o=(O,))` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Affine linear layer (forward)

Matrix multiply plus bias: `y = x @ w + b`.

### `linear(x_bsh: Array, params: LinearParams) -> Array`

| | |
|---|---|
| **`x_bsh`** | `(B, S, I)` |
| **`params.w_io`** | `(I, O)` |
| **`params.b_o`** | `(O,)` |
| **returns** | `(B, S, O)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### LayerNorm param init

Identity scale, zero shift at start.

### `_layer_norm(key: Array, hidden: int) -> LayerNormParams`

| | |
|---|---|
| **`key`** | unused (API symmetry with other inits) |
| **`hidden`** | hidden size `H` |
| **returns** | `LayerNormParams(gamma_h=(H,), beta_h=(H,))` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Layer normalization (forward)

Normalize over the last axis; apply learned scale and shift.

### `layer_norm(x_bsh: Array, params: LayerNormParams) -> Array`

| | |
|---|---|
| **`x_bsh`** | `(B, S, H)` activations |
| **`params`** | `LayerNormParams` |
| **returns** | `(B, S, H)` normalized activations |

**Implement in the code cell below.**


In [ ]:
# your code here


### Multi-head attention params

Four separate Q/K/V/O linear projections.

### `_mha_params(key: Array, hidden: int) -> MultiHeadAttentionParams`

| | |
|---|---|
| **`key`** | split into 4 subkeys |
| **`hidden`** | model width `H` |
| **returns** | `MultiHeadAttentionParams` with four `LinearParams(H, H)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Causal attention mask

Lower-triangular mask so position `t` cannot attend to future tokens.

### `causal_mask(n_seq: int) -> Array`

| | |
|---|---|
| **`n_seq`** | sequence length `S` |
| **returns** | `(S, S)` float mask (1 = keep, 0 = mask out) |

**Implement in the code cell below.**


In [ ]:
# your code here


### Multi-head self-attention

1. Linear projections → Q, K, V each `(B, S, H)`
2. Reshape to `(B, P, S, D)` with `D = H // P`
3. Scaled dot-product attention with causal mask
4. Merge heads → `(B, S, H)` → output projection

**Shapes through attention:** `scores_bpss (B, P, S, S)`, `attn_bpss` same, `out_bpsd (B, P, S, D)`.


### Multi-head attention



### `mha(x_bsh: Array, params: MultiHeadAttentionParams, *, n_heads: int, mask_ss: Array) -> Array`

| | |
|---|---|
| **`x_bsh`** | `(B, S, H)` |
| **`params`** | Q/K/V/O linear params |
| **`n_heads`** | `P` (must divide `H`) |
| **`mask_ss`** | `(S, S)` causal mask |
| **returns** | `(B, S, H)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### MLP block params

Up-project with GELU, then down-project back to hidden size.

### `_mlp_params(key: Array, hidden: int, mlp_mult: int) -> MLPParams`

| | |
|---|---|
| **`key`** | split into 2 subkeys |
| **`hidden`** | `H` |
| **`mlp_mult`** | expansion factor (4 for GPT-2) |
| **returns** | `MLPParams(fc_hf=(H, F), fc_fh=(F, H))` with `F = mlp_mult * H` |

**Implement in the code cell below.**


In [ ]:
# your code here


### GELU activation

GPT-2 uses the tanh approximation of GELU.

### `gelu(x: Array) -> Array`

| | |
|---|---|
| **`x`** | any-shaped array |
| **returns** | same shape as `x` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Feed-forward MLP

Two linear layers with GELU in the middle (4× expansion inside the block).

### `mlp(x_bsh: Array, params: MLPParams) -> Array`

| | |
|---|---|
| **`x_bsh`** | `(B, S, H)` |
| **`params`** | `MLPParams` |
| **returns** | `(B, S, H)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### One transformer block params

Pre-LN block: LN → attention → residual, LN → MLP → residual.

### `_block_params(key: Array, hidden: int, mlp_mult: int) -> BlockParams`

| | |
|---|---|
| **`key`** | split into 4 subkeys (ln1, attn_hh, ln2, mlp_hh) |
| **`hidden`** | `H` |
| **`mlp_mult`** | MLP expansion |
| **returns** | `BlockParams` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Transformer block (forward)

Pre-LN residual block: `x + attn(LN(x))`, then `x + mlp(LN(x))`.

### `block(x_bsh: Array, params: BlockParams, *, n_heads: int, mask_ss: Array) -> Array`

| | |
|---|---|
| **`x_bsh`** | `(B, S, H)` |
| **`params`** | one `BlockParams` |
| **`n_heads`** | attention head count |
| **`mask_ss`** | causal mask `(S, S)` |
| **returns** | `(B, S, H)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Full model init

Token + position embeddings, `n_layers` blocks, final layer norm.

### `init_params(key: Array, config: GPTConfig) -> TransformerParams`

| | |
|---|---|
| **`key`** | split for embed / pos / blocks |
| **`config`** | `GPTConfig` |
| **returns** | `TransformerParams` |

**Implement in the code cell below.**


In [ ]:
# your code here


### Full forward pass

Embed tokens + positions, run all blocks, final LN, **weight-tied** LM head (`logits = x @ embed.T`).

### `forward(params: TransformerParams, tokens_bs: Array, *, config: GPTConfig) -> Array`

| | |
|---|---|
| **`params`** | full model weights |
| **`tokens_bs`** | `(B, S)` int token ids |
| **`config`** | `GPTConfig` (for `n_heads`, `max_ctx`) |
| **returns** | `logits_bsv` with shape `(B, S, V)` |

**Implement in the code cell below.**


In [ ]:
# your code here


## Loss

Next-token prediction: cross-entropy between `logits_bsv` and integer targets `targets_bs`. We use `log_softmax` + `take_along_axis` for numerical stability.


### Cross-entropy from logits



### `cross_entropy_loss(logits_bsv: Array, targets_bs: Array) -> Array`

| | |
|---|---|
| **`logits_bsv`** | `(B, S, V)` unnormalized scores |
| **`targets_bs`** | `(B, S)` int token ids |
| **returns** | scalar mean loss (`Array` rank 0) |

**Implement in the code cell below.**


In [ ]:
# your code here


### Model loss wrapper

Forward pass then cross-entropy — this is what `grad` differentiates.

### `loss(params: TransformerParams, tokens_bs: Array, targets_bs: Array, config: GPTConfig) -> Array`

| | |
|---|---|
| **`params`** | model weights |
| **`tokens_bs`** | `(B, S)` input tokens |
| **`targets_bs`** | `(B, S)` shifted targets (next token) |
| **`config`** | `GPTConfig` |
| **returns** | scalar loss |

**Implement in the code cell below.**


In [ ]:
# your code here


## Optimizer — plain SGD

No Adam, no momentum — just `params - lr * grads` via `jax.tree.map` (Episode 5).


### SGD update

Apply the same learning rate to every leaf in the parameter pytree.

### `sgd_update(params: TransformerParams, grads: TransformerParams, learning_rate: float) -> TransformerParams`

| | |
|---|---|
| **`params`** | current weights |
| **`grads`** | same-structure gradient pytree |
| **`learning_rate`** | scalar step size |
| **returns** | updated `TransformerParams` |

**Implement in the code cell below.**


In [ ]:
# your code here


## Training step

One step: sample a batch → `value_and_grad(loss)` → SGD update. We'll `jit` the step in the demo.


### Random batch (synthetic)

Sample contiguous token windows; inputs are `[:, :-1]`, targets are `[:, 1:]`.

### `sample_batch(key: Array, n_batch: int, n_seq: int, vocab_dim: int) -> tuple[Array, Array]`

| | |
|---|---|
| **`key`** | PRNG key |
| **`n_batch`** | `B` |
| **`n_seq`** | `S` |
| **`vocab_dim`** | `V` |
| **returns** | `(tokens_bs, targets_bs)` each `(B, S)` |

**Implement in the code cell below.**


In [ ]:
# your code here


### One training step



### `train_step(params: TransformerParams, tokens_bs: Array, targets_bs: Array, config: GPTConfig) -> tuple[TransformerParams, Array]`

| | |
|---|---|
| **`params`** | current weights |
| **`tokens_bs`** | `(B, S)` inputs |
| **`targets_bs`** | `(B, S)` targets |
| **`config`** | `GPTConfig` |
| **returns** | `(params, loss_val)` — updated params and scalar loss |

**Implement in the code cell below.**


In [ ]:
# your code here


## Generation

Autoregressive sampling: append one token at a time using `jax.random.categorical` on the last-position logits. Context is truncated to `max_ctx`.


### Autoregressive sampling



### `sample_tokens(params: TransformerParams, prompt_s: Array, *, config: GPTConfig, n_new: int, key: Array) -> Array`

| | |
|---|---|
| **`params`** | trained weights |
| **`prompt_s`** | `(S,)` or `(1, S)` prompt token ids |
| **`config`** | `GPTConfig` |
| **`n_new`** | tokens to generate |
| **`key`** | PRNG for sampling |
| **returns** | `(S + n_new,)` token ids (1-D) |

**Implement in the code cell below.**


In [ ]:
# your code here


## Demo — train on Tiny Shakespeare

Load the corpus from [`data/tiny_shakespeare.txt`](../data/tiny_shakespeare.txt), tokenize with GPT-2 BPE (`tiktoken`), init a **tiny** model, run a short SGD loop, then sample text.



In [ ]:
# your code here


### Corpus batch sampler

Sample random `(B, S+1)` windows from the tokenized corpus; split into inputs and shifted targets.


In [ ]:
# your code here


### Sample text

Even a tiny model and 50 steps won't produce coherent Shakespeare — the point is the **full pipeline** runs end-to-end in pure JAX.


In [ ]:
# your code here


---

## Exercise

1. Print `jax.tree.map(lambda x: x.shape, params.blocks_l[0])` and verify every leaf shape matches the tables above.
2. Change `N_STEPS` to 200 and confirm loss decreases (it may plateau quickly with this tiny model).
3. Implement **`count_params(config: GPTConfig) -> int`** that returns total parameter count **without** calling `init_params` (sum the closed-form sizes for embed, pos, each block, etc.).
4. **Bonus:** add a `@jax.jit`-compiled `forward` and compare step time with `%timeit` before and after (warm up the cache first).


### Exercise 3 — parameter count

Closed form: token embed `V*H`, pos embed `S*H`, each block has `4*(H*H + H)` for Q/K/V/O plus MLP `H*F + F*H` (with `F = mlp_mult * H`), plus two layer norms `2*H` each, final LN `2*H`.


In [ ]:
# your code here
